# Member 1 — Data Cleaning & Preprocessing
## Marketing Mix Modelling and Budget Optimisation
Source: `data/raw/Dataset.xlsx`, sheets `MMM_Data` and `Data_Dictionary`.

This notebook audits the original data, validates types and ranges, reconciles advertising totals, and exports the cleaned data and a column-level dictionary. The source is **synthetic**, not actual company performance or historical economic data. All six advertising channels, including outdoor, are retained. One row represents one national monthly observation.

Run all cells from the project root or this notebook's directory. Dependencies: Python 3.10+, pandas, numpy, openpyxl. No model training, scaling, or encoding is needed for the EDA handoff. Any fitted preprocessing must later use training data only.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/raw/Dataset.xlsx').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Extract the whole submission and run inside its project folder.')
SOURCE = ROOT / 'data/raw/Dataset.xlsx'
PROCESSED = ROOT / 'data/processed'
REPORTS = ROOT / 'reports'
PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
raw = pd.read_excel(SOURCE, sheet_name='MMM_Data')
original_dictionary = pd.read_excel(SOURCE, sheet_name='Data_Dictionary')
df = raw.copy(deep=True)
print('Original shape:', raw.shape)
print(raw.head().to_string(index=False))

Original shape: (36, 25)
     month   country        brand_category region_scope  tv_spend_lkr  facebook_instagram_spend_lkr  google_search_spend_lkr  youtube_spend_lkr  radio_spend_lkr  outdoor_spend_lkr  total_ad_spend_lkr  sales_revenue_lkr  units_sold  impressions   clicks  conversions  average_price_index  promotion_discount_pct  competitor_pressure_index_0_100  economic_sentiment_index_0_100  lkr_usd_exchange_rate  inflation_proxy_index  seasonal_demand_index  public_holiday_count data_type
2023-01-01 Sri Lanka FMCG - Packaged Foods     National       4274762                       3890003                  2467793            1836858          1700676            1813444            15983536           68727157      125509    869144666 30195201      1210051                0.998                    7.27                             49.6                            56.8                 324.42                  1.000                   1.08                     2 SYNTHETIC
2023-02-01 Sri Lanka 

## 1. Understand columns and original types
Primary outcome: `sales_revenue_lkr`. Channel inputs: the six individual advertising spend columns. Context variables include promotions, competitor pressure, seasonality, prices and economic proxies. Impressions, clicks and conversions are retained for exploration; their use as model controls requires care because they can be downstream of advertising. Total advertising spend must not be included alongside every channel in an unregularised linear model because it is their sum.

In [2]:
channels = ['tv_spend_lkr', 'facebook_instagram_spend_lkr',
            'google_search_spend_lkr', 'youtube_spend_lkr',
            'radio_spend_lkr', 'outdoor_spend_lkr']
text_cols = ['country', 'brand_category', 'region_scope', 'data_type']
numeric_cols = [c for c in df.columns if c not in text_cols + ['month']]
integer_cols = channels + ['total_ad_spend_lkr', 'sales_revenue_lkr',
    'units_sold', 'impressions', 'clicks', 'conversions', 'public_holiday_count']
profile = pd.DataFrame({'column': raw.columns,
    'original_dtype': [str(raw[c].dtype) for c in raw],
    'missing_count': [int(raw[c].isna().sum()) for c in raw],
    'unique_count': [int(raw[c].nunique()) for c in raw]})
print(profile.to_string(index=False))

                         column original_dtype  missing_count  unique_count
                          month         object              0            36
                        country         object              0             1
                 brand_category         object              0             1
                   region_scope         object              0             1
                   tv_spend_lkr          int64              0            36
   facebook_instagram_spend_lkr          int64              0            36
        google_search_spend_lkr          int64              0            36
              youtube_spend_lkr          int64              0            36
                radio_spend_lkr          int64              0            36
              outdoor_spend_lkr          int64              0            36
             total_ad_spend_lkr          int64              0            36
              sales_revenue_lkr          int64              0            36
            

## 2. Normalise types and check missing values
Trim surrounding whitespace in text fields. Parse dates and numeric columns strictly. Invalid entries raise an error rather than being silently replaced. No missing values are present in this source, so no imputation is performed.

In [3]:
for c in text_cols:
    df[c] = df[c].astype('string').str.strip()
    df[c] = df[c].replace('', pd.NA)
df['month'] = pd.to_datetime(df['month'], format='%Y-%m-%d', errors='raise')
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors='raise')
missing = df.isna().sum()
print('Missing values:', int(missing.sum()))
assert missing.sum() == 0, 'Missing values need a documented, column-specific treatment.'
for c in integer_cols:
    assert np.isclose(df[c], np.round(df[c])).all(), f'Non-integer values in {c}'
    df[c] = df[c].astype('int64')
print('Date dtype:', df['month'].dtype)

Missing values: 0
Date dtype: datetime64[ns]


## 3. Duplicates, calendar coverage and valid ranges
Remove exact duplicate rows only if found. Conflicting rows for the same month must be investigated, not arbitrarily dropped. Validate nonnegative spending/counts, percentages and bounded indices, positive price/economic multipliers, and the impressions–clicks–conversions ordering. Extreme observations are retained: a high festive-month sales value is not automatically an error.

In [4]:
duplicates = int(df.duplicated().sum())
df = df.drop_duplicates().sort_values('month').reset_index(drop=True)
assert not df['month'].duplicated().any(), 'Conflicting monthly records.'
assert df['month'].dt.is_month_start.all()
expected_months = pd.date_range(df['month'].min(), df['month'].max(), freq='MS')
assert pd.DatetimeIndex(df['month']).equals(expected_months), 'Missing monthly observations.'
assert (df[numeric_cols] >= 0).all().all()
assert df['promotion_discount_pct'].between(0, 100).all()
for c in ['competitor_pressure_index_0_100', 'economic_sentiment_index_0_100']:
    assert df[c].between(0, 100).all(), c
for c in ['average_price_index', 'lkr_usd_exchange_rate',
          'inflation_proxy_index', 'seasonal_demand_index']:
    assert (df[c] > 0).all(), c
assert (df['clicks'] <= df['impressions']).all()
assert (df['conversions'] <= df['clicks']).all()
assert (df['public_holiday_count'] <= df['month'].dt.days_in_month).all()
for c in text_cols:
    assert df[c].nunique() == 1, f'Unexpected context variation in {c}'
assert df['data_type'].eq('SYNTHETIC').all()
print('Exact duplicate rows removed:', duplicates)
print('Period:', df['month'].min().date(), 'to', df['month'].max().date())
print('Continuous monthly observations:', len(df))
print('Range, context and funnel checks passed.')

Exact duplicate rows removed: 0
Period: 2023-01-01 to 2025-12-01
Continuous monthly observations: 36
Range, context and funnel checks passed.


## 4. Reconcile advertising spend totals
The dictionary defines total spend as the sum of channel spend. Use the six channel amounts as authoritative components and recompute the total. The observed discrepancies are only LKR 1–2 and are compatible with rounding, but the source does not establish their cause. Preserve every original total in an audit CSV, together with the recalculated amount and difference. No channel spending is changed.

In [5]:
reconciliation = pd.DataFrame({'month': df['month'],
    'original_total_ad_spend_lkr': df['total_ad_spend_lkr'],
    'recalculated_total_ad_spend_lkr': df[channels].sum(axis=1)})
reconciliation['difference_lkr'] = (reconciliation['recalculated_total_ad_spend_lkr']
    - reconciliation['original_total_ad_spend_lkr'])
changed = reconciliation['difference_lkr'].ne(0)
print('Rows requiring total reconciliation:', int(changed.sum()))
print(reconciliation.loc[changed].to_string(index=False))
df['total_ad_spend_lkr'] = reconciliation['recalculated_total_ad_spend_lkr'].astype('int64')
assert df['total_ad_spend_lkr'].eq(df[channels].sum(axis=1)).all()
reconciliation.to_csv(REPORTS / 'advertising_spend_reconciliation.csv', index=False, date_format='%Y-%m-%d')

Rows requiring total reconciliation: 16
     month  original_total_ad_spend_lkr  recalculated_total_ad_spend_lkr  difference_lkr
2023-04-01                     19087223                         19087224               1
2023-05-01                     20983564                         20983563              -1
2023-06-01                     22467415                         22467414              -1
2023-09-01                     21984643                         21984644               1
2023-12-01                     21520986                         21520987               1
2024-01-01                     15312309                         15312308              -1
2024-02-01                     17437727                         17437726              -1
2024-06-01                     16491131                         16491130              -1
2024-07-01                     15930647                         15930648               1
2024-09-01                     15835297                         158352

## 5. Review and update the Data Dictionary
The original dictionary has a wildcard `*_spend_lkr` entry covering six channels. Expand it into six explicit entries; this is a clarification of existing documentation. Include all 25 columns, output types, units, roles and validation rules. Numeric discounts remain on a 0–100 percentage scale. CSV has no stored date type; consumers must parse `month` as a date.

In [6]:
lookup = original_dictionary.set_index('column_name')
channel_labels = {'tv_spend_lkr': 'TV', 'facebook_instagram_spend_lkr': 'Facebook / Instagram',
    'google_search_spend_lkr': 'Google Search', 'youtube_spend_lkr': 'YouTube',
    'radio_spend_lkr': 'Radio', 'outdoor_spend_lkr': 'Outdoor'}
dictionary_rows = []
for c in df:
    description = ('Monthly advertising spend on ' + channel_labels[c]
                   if c in channels else str(lookup.loc[c, 'description']))
    units = 'LKR' if c in channels else str(lookup.loc[c, 'units_or_format'])
    rule = 'Nonnegative numeric value'
    role = 'Context / potential control'
    if c in text_cols:
        rule, role = 'Nonempty; constant context in this file', 'Metadata'
    elif c == 'month':
        rule, role = 'Unique month start; continuous monthly coverage', 'Time key'
    elif c in channels:
        rule, role = 'Nonnegative whole LKR', 'Channel input'
    elif c == 'total_ad_spend_lkr':
        rule, role = 'Exact sum of all six channel spend columns', 'Derived total'
    elif c == 'sales_revenue_lkr':
        rule, role = 'Nonnegative whole LKR', 'Primary outcome'
    elif c == 'promotion_discount_pct': rule = '0 to 100; percentage points, not fraction'
    elif c.endswith('_0_100'): rule = '0 to 100 inclusive'
    elif c in ['average_price_index', 'lkr_usd_exchange_rate', 'inflation_proxy_index', 'seasonal_demand_index']:
        rule = 'Strictly positive'
    elif c in integer_cols: rule = 'Nonnegative integer'
    if c in ['impressions', 'clicks', 'conversions']: role = 'Funnel measure; assess leakage / mediator risk'
    if c == 'data_type': rule = 'SYNTHETIC for every row'
    dictionary_rows.append({'column_name': c, 'description': description,
        'units_or_format': units, 'cleaned_dtype': str(df[c].dtype),
        'role': role, 'validation_rule': rule})
dictionary = pd.DataFrame(dictionary_rows)
assert dictionary['column_name'].tolist() == df.columns.tolist()
dictionary.to_csv(PROCESSED / 'data_dictionary.csv', index=False)
print(dictionary.to_string(index=False))

                    column_name                                                         description       units_or_format  cleaned_dtype                                           role                                 validation_rule
                          month                                                    Month start date            YYYY-MM-DD datetime64[ns]                                       Time key Unique month start; continuous monthly coverage
                        country                                                     Country context             Sri Lanka         string                                       Metadata         Nonempty; constant context in this file
                 brand_category                                         Illustrative brand/category FMCG - Packaged Foods         string                                       Metadata         Nonempty; constant context in this file
                   region_scope                                         

## 6. Export and verify the handoff
Keep the 36 observations and all 25 original column names. The only numeric changes are the 16 reconciled totals; dates and text are normalised. No arbitrary outlier removal, scaling, encoding or feature selection is applied. With only 36 monthly synthetic observations, later MMM and optimisation results require cautious interpretation and are not validated real-business budget recommendations.

In [7]:
output = PROCESSED / 'cleaned_data.csv'
df.to_csv(output, index=False, date_format='%Y-%m-%d')
roundtrip = pd.read_csv(output, parse_dates=['month'])
assert roundtrip.shape == raw.shape == (36, 25)
assert roundtrip.isna().sum().sum() == 0
assert roundtrip.duplicated().sum() == 0
assert roundtrip['total_ad_spend_lkr'].eq(roundtrip[channels].sum(axis=1)).all()
for c in numeric_cols:
    if c != 'total_ad_spend_lkr':
        assert np.array_equal(roundtrip[c].to_numpy(), raw[c].to_numpy()), c
print('Verified cleaned CSV:', roundtrip.shape)
print('Missing values: 0; duplicates: 0; unreconciled totals: 0')
print('Dictionary coverage:', len(dictionary), 'of', len(df.columns), 'columns')
print('Saved data/processed/cleaned_data.csv and data/processed/data_dictionary.csv')

Verified cleaned CSV: (36, 25)
Missing values: 0; duplicates: 0; unreconciled totals: 0
Dictionary coverage: 25 of 25 columns
Saved data/processed/cleaned_data.csv and data/processed/data_dictionary.csv


## Cleaning summary
- 36 monthly rows and 25 columns retained; January 2023–December 2025.
- No missing values, exact duplicates, calendar gaps, negative numbers, or violations of the implemented range checks.
- Parsed `month` into datetime and standardised text/integer types.
- Recomputed 16 advertising totals; original values retained in the reconciliation report.
- Expanded the dictionary wildcard to six named channel entries, resulting in 25 documented columns.
- Source workbook preserved unchanged for reproducibility.

Member 2 can load the handoff with `pd.read_csv('data/processed/cleaned_data.csv', parse_dates=['month'])`.